**Libraries**

In [1]:
from singleCAM_IROS._pipeline_support import _handle_dirpaths

from typing import Callable
from pathlib import Path

import numpy as np
from numpy.typing import NDArray 
from scipy.optimize import least_squares

from bloodmoon.coords import pos2shift, shift2angle
from bloodmoon.mask import CodedMaskCamera, codedmask
from bloodmoon.mask import count, decode
from bloodmoon.mask import variance, snratio
from bloodmoon.io import simulation_files
from bloodmoon.optim import model_sky

import darksun as ds

**Analysis Methods**

In [2]:
# def model_sky(
#     camera: CodedMaskCamera,
#     px_shift_x: float,
#     px_shift_y: float,
#     fluence: float,
#     vignetting: bool = True,
#     psfy: bool = True,
# ) -> NDArray:
#     """
#     Generate a model of the reconstructed sky image for a point source.
#     """
#     pxdimy, pxdimx = (
#         camera.specs.mask_deltay / camera.upscale_f.y,
#         camera.specs.mask_deltax / camera.upscale_f.x,
#     )
#     detector = model_shadowgram(
#         camera, px_shift_x * pxdimx, px_shift_y * pxdimy, vignetting, psfy,
#     )
#     return decode(camera, detector * fluence)

In [3]:
def handle_det_spres(dataset: str) -> bool:
    """Handles detector spatial resolution correction."""
    if dataset not in ('detected', 'reconstructed'):
        raise ValueError('Nah-huh...')
    
    return False if dataset == 'detected' else True


def find_candidate(
    sky: NDArray,
    snr: NDArray,
    snr_threshold: int | float,
    batch: int = 1000,
) -> tuple[int, int] | bool:
    """
    Returns the position of a valid IROS candidate inside the sky image.
    """
    reservoir = np.array(
        [np.unravel_index(id_, sky.shape) for id_ in np.argsort(sky, axis=None)[-batch:]]
    )
    for pos in reservoir[::-1]:
        if (snr[*pos] > snr_threshold):
            return tuple(pos)
    return False


def _init_source_model(
    camera: CodedMaskCamera,
    vignetting: bool = True,
    psfy: bool = True,
) -> Callable[[float, float, float], NDArray]:
    """
    A slow, vanilla implementation of the model for both direction and fluence optimization.
    Intended for debugging and benchmarking.

    Args:
        camera: CodedMaskCamera instance containing all geometric parameters
        vignetting: If true, shadowgram model simulates vignetting.
        psfy: If true, the model used for optimization will simulate detector position
        reconstruction effects.

    Returns:
        A Callable, which is the routine for computing the model.
    """

    def f(shift_x: float, shift_y: float, fluence: float) -> NDArray:
        """
        A simple, slow version of the model for both direction and fluence optimization.

        Args:
            shift_x: Source position x-coordinate in sky-shift space (mm)
            shift_y: Source position y-coordinate in sky-shift space (mm)
            fluence: Source intensity/fluence value

        Returns:
            2D array representing the modeled sky reconstruction
        """
        return model_sky(camera, shift_x, shift_y, fluence, vignetting=vignetting, psfy=psfy)
    
    return f


def _init_loss_metric(
    true: NDArray,
    pos: tuple[int, int],
    camera: CodedMaskCamera,
    model_source: Callable[[float, float, float], NDArray],
) -> Callable[
    [tuple[float, float, float]],
    float,
]:
    """
    Initialises the loss.
    """
    cropy, cropx = (
        int(camera.specs.slit_deltay * camera.upscale_f.y / camera.specs.mask_deltay) + camera.upscale_f.y,
        int(camera.specs.slit_deltax * camera.upscale_f.x / camera.specs.mask_deltax) + camera.upscale_f.x,
    )
    i, j = pos
    slicey, slicex = (
        slice(i - cropy, i + cropy + 1),
        slice(j - cropx, j + cropx + 1),
    )
    true_ = true[slicey, slicex]

    def f(args: tuple[float, float, float]) -> float:
        """Loss metric for optimisation."""
        sky = model_source(*args)[slicey, slicex]
        metric_val = np.mean(np.square(sky - true_))
        return metric_val
    
    return f


#def optimise(
#    true: NDArray,
#    arg_sky: tuple[int, int],
#    camera: CodedMaskCamera,
#    vignetting: bool,
#    psfy: bool,
#    verbose: bool = True,
#) -> tuple[float, float, float]:
#    """
#    Source parameters optimisation procedure.
#    """
#    px_dim_x, px_dim_y = (
#        camera.specs.mask_deltax / camera.upscale_f.x,
#        camera.specs.mask_deltay / camera.upscale_f.y,
#    )
#
#    source_model = _init_source_model(camera, vignetting, psfy)
#    loss = _init_loss_metric(true, arg_sky, camera, source_model)
#
#    sx_start, sy_start = pos2shift(camera, *arg_sky)
#    px_sx_start, px_sy_start = sx_start / px_dim_x, sy_start / px_dim_y
#    fluence_start = true[*arg_sky] / 0.9                 # camera coding power (Skinner et al. 2008)
#
#    with ds.timer('Optimising'):
#        results = least_squares(
#            loss,
#            x0=np.array((px_sx_start, px_sy_start, fluence_start)),
#            bounds=[
#                (
#                    max(px_sx_start - 3, -len(camera.bins_sky.x) // 2 + 1),
#                    max(px_sy_start - 3, -len(camera.bins_sky.y) // 2 + 1),
#                    true[*arg_sky],
#                ),
#                (
#                    min(px_sx_start + 3, len(camera.bins_sky.x) // 2 - 1),
#                    min(px_sy_start + 3, len(camera.bins_sky.y) // 2 - 1),
#                    true[*arg_sky] / 0.8,
#                ),
#            ],
#            xtol=1e-6,
#            ftol=1e-5,
#        )
#    # store the final optimized positions and fluence
#    px_sx, px_sy, fluence = map(float, results.x[:3])
#    sx, sy = px_sx * px_dim_x, px_sy * px_dim_y
#
#    # optimization verbose
#    if verbose:
#        print(
#            f'\n'
#            f'## Optimisation Results:\n'
#            f'  - fluence START: {fluence_start}\n'
#            f'  - shifts START (x, y): {sx_start}, {sy_start}\n'
#
#            f'  - fluence OPTIM.: {fluence}\n'
#            f'  - shifts OPTIM. (x, y): {sx}, {sy}\n'
#
#            f'  - fluence GAIN %: {(fluence - fluence_start) * 100 / fluence_start:.3f}\n'
#            f'  - shift_x GAIN %: {(sx - sx_start) * 100 / sx_start:.3f}\n'
#            f'  - shift_y GAIN %: {(sy - sy_start) * 100 / sy_start:.3f}\n'
#        )
#
#    return sx, sy, fluence


def shift2arcmin(camera: CodedMaskCamera, shift: float) -> float:
    """Shift to angular coord in [arcmin] conversion."""
    return shift2angle(camera, shift) * 60


def fluence_error(observed: float, true: float) -> float:
    """Returns the percentage error on the observed fluence."""
    return (observed - true) * 100 / true

In [ ]:
#from bloodmoon.optim import _ModelShiftFluence
from bloodmoon.optim import _Loss
from scipy.optimize import minimize, curve_fit

def _ModelShiftFluence(
    camera: CodedMaskCamera,
    pos: tuple[int, int],
    vignetting: bool = True,
    psfy: bool = True,
) -> Callable[[NDArray, float, float, float], NDArray]:
    """
    A slow, vanilla implementation of the model for both direction and fluence optimization.
    Intended for debugging and benchmarking.

    Args:
        camera: CodedMaskCamera instance containing all geometric parameters
        pos: tuple of row, col indexes indicating the source peak position. The source
        sky image is cropped around `pos`.
        vignetting: If true, shadowgram model simulates vignetting.
        psfy: If true, the model used for optimization will simulate detector position
        reconstruction effects.

    Returns:
        A Callable, which is the routine for computing the model.
    """
    cropy, cropx = (
        int(camera.specs.slit_deltay * camera.upscale_f.y / camera.specs.mask_deltay) + camera.upscale_f.y,
        int(camera.specs.slit_deltax * camera.upscale_f.x / camera.specs.mask_deltax) + camera.upscale_f.x,
    )
    i, j = pos
    slicey, slicex = (
        slice(i - cropy, i + cropy + 1),
        slice(j - cropx, j + cropx + 1),
    )

    def f(x: NDArray, shift_x: float, shift_y: float, fluence: float) -> NDArray:
        """
        A simple, slow version of the model for both direction and fluence optimization.

        Args:
            x: Placeholder for independent variable as in `curve_fit` doc
            shift_x: Source position x-coordinate in sky-shift space (mm)
            shift_y: Source position y-coordinate in sky-shift space (mm)
            fluence: Source intensity/fluence value

        Returns:
            Flattened and cropped 2D source-modeled sky image
        """
        modeled = model_sky(camera, shift_x, shift_y, fluence, vignetting, psfy)
        flattened = modeled[slicey, slicex].flatten()
        return flattened
    
    return f


def optimize(
    camera: CodedMaskCamera,
    sky: NDArray,
    arg_sky: tuple[int, int],
    vignetting: bool = True,
    psfy: bool = True,
    verbose: bool = True,
) -> tuple[float, float, float]:
    """
    Performs the optimization to fit a point source model to sky image data.

    This function performs the optimization by simultaneously fit the candidate
    position and fluence. The starting position is inferred from the candidate
    pixel position, while the starting fluence is represented by the counts at
    the candidate extracted pixel indexes.

    Args:
        camera: CodedMaskCamera instance containing detector and mask parameters
        sky: 2D array of the reconstructed sky image to fit
        arg_sky: Initial guess for source position as (row, col) indices
        vignetting: If true, the model used for optimization will simulate vignetting.
        psfy: If true, the model used for optimization will simulate detector position
        reconstruction effects.

    Returns:
        Tuple containing the best-fit parameters `(x, y, fluence)` where:
                - x, y are the optimized sky-shift coordinates
                - fluence is the optimized source intensity

    Notes:
        - Bounds are set based on initial guess and physical constraints
    """
    model_shift_flux = _ModelShiftFluence(camera, arg_sky, vignetting, psfy)
    # loss = _Loss(model_shift_flux)

    px_dim_x, px_dim_y = (
        camera.specs.mask_deltax / camera.upscale_f.x,
        camera.specs.mask_deltay / camera.upscale_f.y,
    )
    cropy, cropx = (
        int(camera.specs.slit_deltay * camera.upscale_f.y / camera.specs.mask_deltay) + camera.upscale_f.y,
        int(camera.specs.slit_deltax * camera.upscale_f.x / camera.specs.mask_deltax) + camera.upscale_f.x,
    )
    slicey, slicex = (
        slice(arg_sky[0] - cropy, arg_sky[0] + cropy + 1),
        slice(arg_sky[1] - cropx, arg_sky[1] + cropx + 1),
    )

    sx_start, sy_start = pos2shift(camera, *arg_sky)
    sky_peak = sky[*arg_sky]
    fluence_start = (
        sky_peak / 0.85 if psfy else sky_peak
    )
    
    with ds.timer('Optimisation'):
        results, _ = curve_fit(
            model_shift_flux,
            xdata=np.arange((2 * cropy + 1) * (2 * cropx + 1)),
            ydata=sky[slicey, slicex].flatten(),
            p0=[sx_start, sy_start, fluence_start],
            bounds=[
                (
                    max(sx_start - 3 * px_dim_x, camera.bins_sky.x[0]),
                    max(sy_start - 3 * px_dim_y, camera.bins_sky.y[0]),
                    sky_peak,
                ),
                (
                    min(sx_start + 3 * px_dim_x, camera.bins_sky.x[-1]),
                    min(sy_start + 3 * px_dim_y, camera.bins_sky.y[-1]),
                    1.25 * sky_peak,
                ),
            ],
        )
    # store the final optimized positions and fluence
    sx, sy, fluence = map(float, results)

    #with ds.timer('Optimisation'):
    #    results = minimize(
    #        lambda args: loss((args[0], args[1], args[2]), sky, arg_sky, camera),
    #        x0=np.array((sx_start, sy_start, fluence_start)),
    #        method="Nelder-Mead",
    #        bounds=[
    #            (
    #                max(sx_start - 3 * px_dim_x, camera.bins_sky.x[0]),
    #                min(sx_start + 3 * px_dim_x, camera.bins_sky.x[-1]),
    #            ),
    #            (
    #                max(sy_start - 3 * px_dim_y, camera.bins_sky.y[0]),
    #                min(sy_start + 3 * px_dim_y, camera.bins_sky.y[-1]),
    #            ),
    #            (0.75 * fluence_start, 1.25 * fluence_start),
    #        ],
    #        options={
    #            "xatol": 1e-6,
    #        },
    #        )
    ## store the final optimized positions and fluence.
    #sx, sy, fluence = map(float, results.x[:3])

    if verbose:
        print(
            f'\n'
            f'## Optimisation Results:\n'
            f'  - fluence START: {fluence_start}\n'
            f'  - shifts START (x, y): {sx_start}, {sy_start}\n'

            f'  - fluence OPTIM.: {fluence}\n'
            f'  - shifts OPTIM. (x, y): {sx}, {sy}\n'

            f'  - fluence GAIN %: {(fluence - fluence_start) * 100 / fluence_start:.3f}\n'
            f'  - shift_x GAIN %: {np.sign(sx_start) * (sx - sx_start) * 100 / sx_start:.3f}\n'
            f'  - shift_y GAIN %: {np.sign(sy_start) * (sy - sy_start) * 100 / sy_start:.3f}\n'
        )

    return sx, sy, fluence

**Mask and Data Specifics**

In [5]:
MASK_FITS: str = "wfm_mask_NTHT_20250725.fits"

SKYFIELD: str = "GalacticCentre"
DATA_FITS: str = "galctr_rxte-sax_2-50keV_mask_050_1040x17_opaquemask_infdet"

ID_CAMERA_A: str = "cam1a"
ID_CAMERA_B: str = "cam1b"
DATASET: str = "reconstructed"

UPS_X: int = 2
UPS_Y: int = 1

VIGNETTING: bool = True
PSFY: bool = handle_det_spres(DATASET)

In [6]:
dataIsLoaded: bool = False

**Load Mask and Data**

In [7]:
mask_path, simul_data, _ = _handle_dirpaths(
        mask=MASK_FITS,
        skyfield=SKYFIELD,
        simul=DATA_FITS,
    )
wfm: CodedMaskCamera = codedmask(mask_path, UPS_X, UPS_Y)
filepaths: dict[str, dict[str, Path]] = simulation_files(simul_data)
sdlA = ds.get_data(filepaths[ID_CAMERA_A][DATASET])
sdlB = ds.get_data(filepaths[ID_CAMERA_B][DATASET])

In [8]:
if not dataIsLoaded:
    detector_camA = count(wfm, sdlA.DLdata)[0]
    true_sky_camA = decode(wfm, detector_camA)
    varmap_camA = variance(wfm, detector_camA)
    snr_camA = snratio(true_sky_camA, varmap_camA)

    detector_camB = count(wfm, sdlB.DLdata)[0]
    true_sky_camB = decode(wfm, detector_camB)
    varmap_camB = variance(wfm, detector_camB)
    snr_camB = snratio(true_sky_camB, varmap_camB)

    dataIsLoaded = True

## USING BULK MASK with 1.5 mm cover ##


**Analysis for LEM-X Camera A**

In [9]:
arg_sky_camA = find_candidate(true_sky_camA, snr_camA, snr_threshold=5.0)
sxA, syA, fA = optimize(wfm, true_sky_camA, arg_sky_camA, VIGNETTING, PSFY)

    # Starting 'Optimisation' at 16:21:01.
    # Finished 'Optimisation' in 00h:00m:4.737s.

[4.39108918e+01 7.80445829e+01 1.05244545e+06]

## Optimisation Results:
  - fluence START: 1135364.2874728004
  - shifts START (x, y): 43.875, 78.0
  - fluence OPTIM.: 1052445.4508142471
  - shifts OPTIM. (x, y): 43.91089183770607, 78.04458293946902
  - fluence GAIN %: -7.303
  - shift_x GAIN %: 0.082
  - shift_y GAIN %: 0.057



In [10]:
true_params_camA = (
    43.909402530749, 78.04581352417, (1064657.0 if DATASET == 'reconstructed' else 1073525.0),
)

print(
    f'## {DATASET.capitalize()} Dataset\n\n'

    f'# Fit {ID_CAMERA_A.upper()}\n'
    f'  - coords residues along fine dir: {shift2arcmin(wfm, sxA - true_params_camA[0])} [arcmin]\n'
    f'  - coords residues along coarse dir: {shift2arcmin(wfm, syA - true_params_camA[1])} [arcmin]\n'
    f'  - fluence residues: {fluence_error(fA, true_params_camA[2])} [%]\n'
)

## Reconstructed Dataset

# Fit CAM1A
  - coords residues along fine dir: 0.025214775583745415 [arcmin]
  - coords residues along coarse dir: -0.020834467283473785 [arcmin]
  - fluence residues: -1.1469937440652593 [%]



**Analysis for LEM-X Camera B**

In [11]:
arg_sky_camB = find_candidate(true_sky_camB, snr_camB, snr_threshold=5.0)
sxB, syB, fB = optimize(wfm, true_sky_camB, arg_sky_camB, VIGNETTING, PSFY)

    # Starting 'Optimisation' at 16:21:05.
    # Finished 'Optimisation' in 00h:00m:4.843s.

[ 7.80459612e+01 -4.39377033e+01  9.47321843e+05]

## Optimisation Results:
  - fluence START: 1028076.4250290795
  - shifts START (x, y): 78.0, -44.0
  - fluence OPTIM.: 947321.8431266105
  - shifts OPTIM. (x, y): 78.04596118077798, -43.93770326172701
  - fluence GAIN %: -7.855
  - shift_x GAIN %: 0.059
  - shift_y GAIN %: 0.142



In [12]:
true_params_camB = (
    78.04581352417, -43.909402530749, (952281.0 if DATASET == 'reconstructed' else 955717.0),
)

print(
    f'## {DATASET.capitalize()} Dataset\n\n'

    f'# Fit {ID_CAMERA_B.upper()}\n'
    f'  - coords residues along fine dir: {shift2arcmin(wfm, sxB - true_params_camB[0])} [arcmin]\n'
    f'  - coords residues along coarse dir: {shift2arcmin(wfm, syB - true_params_camB[1])} [arcmin]\n'
    f'  - fluence residues: {fluence_error(fB, true_params_camB[2])} [%]\n'
)

## Reconstructed Dataset

# Fit CAM1B
  - coords residues along fine dir: 0.0024999065612607736 [arcmin]
  - coords residues along coarse dir: -0.47914674169174937 [arcmin]
  - fluence residues: -0.5207661261108325 [%]

